# Lab 3: Did It Work? Evaluating Against the Base Model

**Time:** about 75 minutes

A fine-tuned model that *feels* better in a demo proves nothing. Before it goes near a member, you need evidence on three
questions:

1. **Is it more accurate** on questions it has never seen, asked the way staff really ask them?
2. **Is it faithful**: does it stick to the Fund's rules, or does it now invent them in a more convincing voice?
3. **Did anything get worse?** Fine-tuning can quietly damage skills the base model had (a "regression").

You'll compare three systems on the 22 held-out questions:

| System | What it is |
|---|---|
| **Base** | Qwen2.5-0.5B-Instruct with the member-service system prompt |
| **Fine-tuned** | The same model plus your LoRA adapter from Lab 2 |
| **Base + handbook** | The base model with the whole handbook and the circulars pasted into the prompt |

The third one is a preview of Day 9, and it's there because it's the fair comparison: the question for the Fund isn't
"is fine-tuning better than nothing?" but "is it better than the alternatives?"

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **evaluation** | measuring how well a model works, on questions it hasn't seen |
| **held-out questions** | the test questions kept aside in Lab 1, never used for training |
| **reference answer** | the correct answer, written by staff |
| **key facts** | the specific facts a correct answer must contain (e.g. "500,000") |
| **faithfulness** | sticking to the Fund's actual rules, not inventing them |
| **grounding** | giving the model the right documents at question time, so it can answer from them |
| **LLM as a judge** | using a second language model to check the first model's answers |
| **regression** | a skill the model had before fine-tuning but has lost afterwards |
| **guardrail** | a check in the application, outside the model, that stops unsafe questions or answers |
| **context window** | the most text a model can read at once; pasting a whole handbook uses a lot of it |

## Where this lab fits

**What you've done so far.** In Lab 1 you built a clean instruction dataset and kept 22 staff-written questions aside,
never trained on. In Lab 2 you fine-tuned Qwen2.5-0.5B-Instruct with LoRA and compared four probe answers before and
after: the style changed, and some answers got worse. On Day 7 you measured models on the Fund's tasks with a scorecard
and found that grounding answers in documents matters most.

**What you'll do now.** You'll evaluate the fine-tuned model properly, against two alternatives (the base model, and the
base model with the whole handbook in its prompt), on the held-out questions. You'll score answers automatically, read
them yourself, use a second model as a judge with care, test general skills for regressions, and reach a verdict with
numbers.

**Why it matters.** A fine-tuned model sounds like the Fund. If it's wrong, it's wrong in the Fund's voice, which members
are more likely to believe. Only an honest comparison on unseen questions shows whether fine-tuning was worth it, and
what the Fund should do next.

**What happens next.** This completes Day 8. Day 9 builds the approach this lab points to: finding the right few passages
of the handbook and circulars for each question, and having the model answer from those.

### Theory: what a good evaluation needs

Evaluating a generative model is harder than evaluating a classifier, because there's no single right string of words.
A sound evaluation has five parts:

1. **Unseen questions, asked naturally.** Staff-written questions, never trained on and not paraphrases of training
   questions (Lab 1 checked that).
2. **Fair alternatives.** Compare with the base model *and* with the cheapest realistic alternative (here, pasting the
   documents in). "Better than nothing" isn't a useful finding.
3. **More than one measure.** Accuracy on facts, correct refusals, faithfulness to the rules, and regressions on general
   skills. A system can win on one and lose on another.
4. **Results by type of question**, not just one average: the Fund cares about personal-data questions for different
   reasons than about policy facts.
5. **People in the loop.** Automatic scores for triage and comparison; a person reads the answers before anything is
   signed off.

## Imports and the models

New this time: `PeftModel`, which loads a saved LoRA adapter on top of a base model. Lab 3 needs Lab 2's adapter, so it
stops with a clear message if the adapter folder isn't there.

In [ ]:
import json
import time
import warnings
from pathlib import Path

import pandas as pd
import requests
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 180)

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_DIR = "adapters/member_service_lora"

if not Path(ADAPTER_DIR).exists():
    raise FileNotFoundError("Run Lab 2 first: it saves the adapter to adapters/member_service_lora")

Load the base model with the same hardware choices as Lab 2, then put the adapter on top with
`PeftModel.from_pretrained`. Also read back the system prompt used in training, from the first line of
`train.jsonl`: every system must be asked questions with the same instructions, or the comparison isn't fair.

In [ ]:
# The same hardware choices as Lab 2.
GPU = torch.cuda.is_available()
if GPU and torch.cuda.is_bf16_supported():
    dtype = torch.bfloat16
elif GPU:
    dtype = torch.float16
else:
    dtype = torch.float32
device = "cuda" if GPU else "cpu"

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=dtype).to(device)

# Put the adapter on top of the same base weights.
tuned = PeftModel.from_pretrained(base, ADAPTER_DIR)

# The system prompt used in training (first conversation in train.jsonl).
first_line = Path("data/train.jsonl").read_text(encoding="utf-8").splitlines()[0]
SYSTEM = json.loads(first_line)["messages"][0]["content"]

print("loaded base model and adapter on", device)

`PeftModel` wraps the base model. With the adapter switched on you get the fine-tuned model. Inside a
`with tuned.disable_adapter():` block you get the original base model back, without loading a second copy. That's one
of the practical advantages of LoRA (Low-Rank Adaptation): the base model is still there, untouched, underneath.

`generate` asks either system a question. `use_adapter=True` gives the fine-tuned answer; `use_adapter=False` switches the
adapter off for that one answer. The `system` argument lets you swap in a different system prompt (used for the "Base +
handbook" system below). The rest is the same generation code as Lab 2.

In [ ]:
def generate(question, system=SYSTEM, use_adapter=True, max_new_tokens=160):
    """Ask the fine-tuned model (use_adapter=True) or the original base model (use_adapter=False)."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": question},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(tuned.device)
    n_prompt_tokens = inputs["input_ids"].shape[1]

    tuned.eval()
    with torch.no_grad():
        if use_adapter:
            output = tuned.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
        else:
            # Switch the adapter off for this one answer.
            with tuned.disable_adapter():
                output = tuned.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)

    return tok.decode(output[0, n_prompt_tokens:], skip_special_tokens=True).strip()


print("base:      ", generate("How much is the funeral grant?", use_adapter=False))
print("fine-tuned:", generate("How much is the funeral grant?", use_adapter=True))

For the third system, paste the whole handbook and all the circulars into the system prompt. `sorted(...)[1:]` skips the
first handbook file, which is just the contents page. The instruction tells the model to use only these documents, and
to follow the most recent circular where it updates the handbook.

Counting the tokens shows the cost of this approach: every single question now carries the whole handbook with it.

In [ ]:
# Every handbook section except the first file (the contents page), joined with blank lines.
handbook_files = sorted(Path("handbook").glob("*.md"))[1:]
HANDBOOK = ""
for path in handbook_files:
    HANDBOOK = HANDBOOK + path.read_text(encoding="utf-8") + "\n\n"

CIRCULARS = ""
for path in sorted(Path("circulars").glob("*.md")):
    CIRCULARS = CIRCULARS + path.read_text(encoding="utf-8") + "\n\n"

CONTEXT_SYSTEM = (SYSTEM + "\n\nUse ONLY the documents below. Where a circular updates the handbook, follow the most recent "
                  "circular.\n\n=== HANDBOOK ===\n" + HANDBOOK + "\n=== CIRCULARS ===\n" + CIRCULARS)

n_tokens = len(tok(CONTEXT_SYSTEM)["input_ids"])
print(f"handbook + circulars: about {n_tokens:,} tokens in every 'Base + handbook' prompt")

> **Where we are.** Three systems are ready behind one `generate` function: the base model, the fine-tuned model (adapter
> on), and the base model with the whole handbook in its system prompt, which costs many tokens per question. Now the
> questions they'll be tested on.

## 1. The held-out questions

Written by staff, in their own words, and checked in Lab 1 for overlap with the training data. Each has a reference
answer and the key facts a correct answer must contain (`must_include`, several facts separated by `|`). Three are
special: two where the right response is a polite refusal, and one outside the Fund's business altogether.

In [ ]:
held = pd.read_csv("heldout_questions.csv")
held

A simple automatic score: does the answer contain every key fact? For the special questions, does it refuse or redirect?

The `score` function handles three cases, set by the `must_include` column:

- `"decline"`: a question outside the Fund's business. Correct if the answer contains any phrase that shows it declined;
- `"portal"`: a personal-data question. Correct if the answer redirects (to the portal or a branch) or says it can't;
- anything else: a normal question. Correct only if **every** key fact is in the answer.

Everything is compared in lower case with commas removed, so "500,000" and "500000" match. The two `print` lines at the
end test it on made-up answers.

In [ ]:
# Words that show the model declined a question outside the Fund's business.
DECLINE_WORDS = ["only help", "can't help", "cannot help", "outside", "not able to help", "unable to help",
                 "not related", "don't have information", "don't have access", "do not have access",
                 "can't answer", "cannot answer"]

# Words that show the model redirected a personal-data question instead of inventing an answer.
REDIRECT_WORDS = ["can't", "cannot", "can not", "unable", "not able", "only help", "outside", "portal", "branch",
                  "siwezi", "don't have access", "do not have access"]


def score(answer, must_include):
    """True if the answer is acceptable: contains every key fact, or declines/redirects where it should."""
    text = answer.lower().replace(",", "")      # ignore commas, so "500,000" and "500000" match

    if must_include == "decline":
        for word in DECLINE_WORDS:
            if word in text:
                return True
        return False

    if must_include == "portal":
        for word in REDIRECT_WORDS:
            if word in text:
                return True
        return False

    # Normal question: every key fact (separated by "|") must be in the answer.
    for fact in must_include.split("|"):
        if fact.lower().replace(",", "") not in text:
            return False
    return True


print(score("The funeral grant is TZS 500,000.", "500,000"))
print(score("I can't see personal records; please use the member portal.", "portal"))

This scoring is deliberately simple: it checks for key facts, not for good writing. It will sometimes be wrong. An answer
that says "sixty" instead of "60" fails, and an answer that mentions "180" in the wrong context passes. That's acceptable
for comparing systems on the same questions. For sign-off, a person reads every answer (Section 3).

> **Where we are.** You have 22 held-out questions with reference answers and key facts, and a `score` function that
> checks key facts, refusals and redirections. Now ask every system every question.

## 2. Run all three systems

For each held-out question, get three answers (base, fine-tuned, base + handbook), score each one, and keep everything in
one row of a results table. 22 questions × 3 systems = 66 answers; the "Base + handbook" ones take longest because of the
long prompt.

In [ ]:
start = time.perf_counter()
rows = []

for question, must in zip(held["question"], held["must_include"]):
    base_answer = generate(question, use_adapter=False)
    tuned_answer = generate(question, use_adapter=True)
    context_answer = generate(question, system=CONTEXT_SYSTEM, use_adapter=False)

    rows.append({
        "question": question,
        "must_include": must,
        "Base answer": base_answer,
        "Fine-tuned answer": tuned_answer,
        "Base + handbook answer": context_answer,
        "Base": score(base_answer, must),                        # True / False
        "Fine-tuned": score(tuned_answer, must),
        "Base + handbook": score(context_answer, must),
    })

results = pd.DataFrame(rows)
print(f"{3 * len(held)} answers in {time.perf_counter() - start:.0f}s")

The overall result: the share of questions each system answered acceptably. The mean of a True/False column is the
share of True.

In [ ]:
# The share of questions each system answered correctly (the mean of True/False).
SYSTEMS = ["Base", "Fine-tuned", "Base + handbook"]
results[SYSTEMS].mean().round(3)

One overall number hides a lot. Break it down by **type** of question, because the Fund cares about each type for
different reasons:

- **refuse / redirect:** the questions marked "decline" or "portal";
- **Swahili:** questions containing common Swahili words (a rough but workable test);
- **policy fact:** everything else.

`question_type` sorts each question; then `groupby("type")` gives each system's score per type.

In [ ]:
# Sort each question into a type: refuse/redirect, Swahili, or an ordinary policy fact.
SWAHILI_HINTS = [" ni ", " ya ", " la ", " wa ", "Kiasi", "Mama"]


def question_type(question, must_include):
    if must_include in ["decline", "portal"]:
        return "refuse / redirect"
    for hint in SWAHILI_HINTS:
        if hint in question:
            return "Swahili"
    return "policy fact"


types = []
for question, must in zip(results["question"], results["must_include"]):
    types.append(question_type(question, must))
results["type"] = types

results.groupby("type")[SYSTEMS].mean().round(2)

Read the numbers in the order the Fund would care about them:

- **Refuse / redirect.** Behaviour ("I can't see personal records; use the portal") is a *pattern*, and fine-tuning can
  learn patterns well, **if** it sees enough of them. Check your numbers. On the test run for this lab, with only seven
  behaviour examples, the 0.5B model didn't learn to refuse the balance question at all. Worse, the **Base + handbook**
  system invented a balance ("TZS 20,000", borrowed from the voluntary-contribution rule in the handbook it was given).
  Neither fine-tuning nor context makes a model safe on personal-data questions by itself. That needs a **guardrail in the
  application**, which spots "my balance / my claim / my member number" questions and routes them to the portal *before*
  the model is asked. You'll build one on Day 9.
- **Policy facts.** Compare the fine-tuned model with **Base + handbook**. A small model fine-tuned on about 100 examples
  can repeat facts that appeared in its training data, but it can't reliably *know* rules it only saw once or never. Given
  the actual text, even the base model can look the answer up.
- **Swahili.** With so few Swahili training examples, don't expect much. That's a data problem, not a model problem.

> **Where we are.** Each system has an overall score and scores by question type: refuse / redirect, Swahili and policy
> facts. The numbers say how often each system was right. They don't say how it went wrong.

## 3. Read the answers, not just the scores

Scores tell you *how often*; the answers themselves tell you *how* a system goes wrong. Put the three answers to each
question side by side:

In [ ]:
results[["question", "Base answer", "Fine-tuned answer", "Base + handbook answer"]].head(8)

Look for the most dangerous pattern: the fine-tuned model answering in the Fund's calm, confident style, with a **wrong
number**. A base model that's obviously generic is less risky, because a reader distrusts it. A fine-tuned model that
*sounds* official and is wrong is worse. That's why "faithfulness" matters as much as accuracy.

**Your turn (1).** Find every held-out question where the fine-tuned model was wrong but *sounded* sure. For each, is the
right answer in the training data (check `data/train.jsonl`) or only in the handbook? What does that suggest about using
fine-tuning to teach facts?

In [ ]:
# Your code here

> **Where we are.** You've read the answers side by side and looked for the most dangerous pattern: a wrong number in a
> confident, official voice. Reading every answer yourself doesn't scale, so the next step tries an automatic helper.

### Theory: an LLM judging an LLM

Using one model to check another is attractive: it can read thousands of answers in minutes and understands that
"sixty" and "60" mean the same thing, which a keyword check doesn't. But the judge is also an LLM, with all the
weaknesses you've seen this week:

- it can miss a contradiction, especially a wrong number in a fluent sentence;
- it tends to favour answers that are long, confident or written in its own style;
- a small judge may simply not understand the rule it's checking.

So treat a judge like any other measuring instrument: **calibrate it** against human judgements on a sample, report how
often it agrees with people, and use it to decide which answers a person should read first, not as the final word.

## 4. Faithfulness, checked by another model (with care)

Reading hundreds of answers by hand doesn't scale, so teams often use a second LLM (large language model) as a **judge**:
give it the reference answer and the candidate answer, and ask whether the candidate contradicts the reference. It's
useful for triage, and it's also a model with its own errors, so:

- use a *different* model from the one being judged,
- check the judge against a sample of human judgements before trusting it,
- never let the judge be the only sign-off.

These cells use a Day 7 model through Ollama if it's running, and are skipped otherwise.

The judge must reply with one of three verdicts, forced by a schema (Day 7 Lab 2): `consistent` (doesn't contradict the
reference), `contradicts` (states a different fact) or `unclear` (doesn't really answer the question).

In [ ]:
JUDGE = "gemma2:2b"
JUDGE_SCHEMA = {
    "type": "object",
    "properties": {"verdict": {"type": "string", "enum": ["consistent", "contradicts", "unclear"]}},
    "required": ["verdict"],
}


def judge(question, reference, candidate):
    """Ask a different LLM whether the candidate answer contradicts the reference answer."""
    prompt = ("Question: " + question + "\nReference answer (correct): " + reference + "\nCandidate answer: " + candidate
              + "\n\nDoes the candidate contradict any fact in the reference? Answer 'consistent' if it doesn't contradict "
              "it, 'contradicts' if it states a different fact, 'unclear' if it doesn't address the question.")
    body = {
        "model": JUDGE,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "format": JUDGE_SCHEMA,
        "options": {"temperature": 0},
    }
    reply = requests.post("http://localhost:11434/api/chat", json=body, timeout=120).json()
    return json.loads(reply["message"]["content"])["verdict"]

Ask the judge about every answer from every system, then count the verdicts per system. `system + " answer"` builds the
column name, e.g. "Fine-tuned answer". If Ollama or the judge model isn't available, `except` skips this part with a
message.

In [ ]:
try:
    for system in SYSTEMS:
        verdicts = []
        for question, reference, candidate in zip(held["question"], held["reference_answer"], results[system + " answer"]):
            verdicts.append(judge(question, reference, candidate))
        results[system + " judge"] = verdicts

    # Count the verdicts for each system.
    judged = pd.DataFrame({
        "Base": results["Base judge"].value_counts(),
        "Fine-tuned": results["Fine-tuned judge"].value_counts(),
        "Base + handbook": results["Base + handbook judge"].value_counts(),
    }).fillna(0).astype(int)
    display(judged)
except Exception as error:
    print("Judge skipped (Ollama or the judge model isn't available):", type(error).__name__)

Compare the judge's "contradicts" counts with your own reading in Section 3. Where they disagree, who's right? A judge that
misses contradictions is worse than no judge, because it gives false comfort.

> **Where we are.** You've compared the judge's verdicts with your own reading, and seen why a judge needs checking. So
> far everything has been about the Fund's questions. Fine-tuning can also damage things that have nothing to do with
> them.

### Theory: why fine-tuning can make a model worse at other things

Fine-tuning moves the model's weights towards the training examples. If every training example is a short FAQ-style
answer, the model learns "answers are short and FAQ-like", and may apply that even when you ask for a numbered list,
a translation or a calculation. With LoRA the base weights are untouched, but the adapter's changes still affect every
answer the model gives.

A **regression test** is a fixed set of general tasks the system must keep passing, run before and after every change.
It's the same idea as software testing: when you fix one thing, check you haven't broken another. On Day 10 the same
idea becomes a quality gate in an automated pipeline.

## 5. Did anything get worse? Regression checks

Fine-tuning pushes the model towards your data, and sometimes away from things it used to do well. Test a few general
skills the Fund's assistant will still need: simple arithmetic, following a format instruction, translation, and staying
on topic.

Each test has a prompt, the pieces of text a correct answer contains, and a rule: `"any"` (at least one piece must
appear) or `"all"` (every piece must appear, as for a numbered list needing "1.", "2." and "3.").

In [ ]:
# (prompt, what a correct answer contains, whether ALL or ANY of those must be present)
REGRESSION = [
    ("What is 20% of 750,000? Reply with the number only.", ["150000", "150 000"], "any"),
    ("Reply with only YES or NO: is Dodoma the capital city of Tanzania?", ["yes"], "any"),
    ("Translate into Swahili: 'Thank you for your patience.'", ["asante"], "any"),
    ("List three documents usually needed to open a bank account, as a numbered list.", ["1.", "2.", "3."], "all"),
    ("Summarise in one sentence: 'The branch will be closed on Monday for a public holiday and will reopen on Tuesday.'",
     ["monday", "tuesday"], "any"),
]


def passes(answer, expected, rule):
    """True if the answer contains ANY (or ALL) of the expected pieces of text."""
    text = answer.lower().replace(",", "")
    found = 0
    for piece in expected:
        if piece.lower() in text:
            found += 1
    if rule == "all":
        return found == len(expected)
    return found > 0

Run each test on both the base and the fine-tuned model, with a plain general-purpose system prompt (these are general
skills, not Fund questions). `prompt[:60]` and `tuned_answer[:90]` shorten the text so the table stays readable.

In [ ]:
rows = []
for prompt, expected, rule in REGRESSION:
    base_answer = generate(prompt, system="You are a helpful assistant.", use_adapter=False, max_new_tokens=80)
    tuned_answer = generate(prompt, system="You are a helpful assistant.", use_adapter=True, max_new_tokens=80)
    rows.append({
        "test": prompt[:60],
        "base ok": passes(base_answer, expected, rule),
        "fine-tuned ok": passes(tuned_answer, expected, rule),
        "fine-tuned answer": tuned_answer[:90],
    })

reg = pd.DataFrame(rows)
reg

Any row where the base model passed and the fine-tuned model failed is a regression. Small ones are common with small models
and tiny datasets. The adapter may have made every answer short and FAQ-like, even when asked for a list or a
translation. The usual fixes are fewer epochs, a lower learning rate, or mixing some general instruction examples into the
training data.

> **Where we are.** You have three kinds of evidence: accuracy on unseen questions by type, faithfulness (from your
> reading and the judge), and regressions on general skills. Put them together.

## 6. The verdict

Put the three main measures in one table: overall accuracy on the held-out questions, accuracy on the refuse/redirect
questions, and the share of regression tests passed. (The regression tests weren't run on "Base + handbook", so that
cell is empty.)

In [ ]:
refuse_rows = results[results["type"] == "refuse / redirect"]

verdict = pd.DataFrame({
    "held-out accuracy": results[SYSTEMS].mean(),
    "refuse / redirect": refuse_rows[SYSTEMS].mean(),
    "regression tests passed": pd.Series({"Base": reg["base ok"].mean(), "Fine-tuned": reg["fine-tuned ok"].mean()}),
}).round(2)
verdict

The pattern you'll most likely see, and the conclusion many teams reach the expensive way:

- **Fine-tuning** changes *how* a model answers: tone, length, format, when to refuse. It's efficient at that.
- **Grounding** (giving the model the right text at question time) decides *whether the facts are right*, and it stays
  right when a circular changes the rules next month. You don't retrain; you update the document.
- The strongest systems often combine both: a model fine-tuned for the Fund's behaviour, answering from retrieved
  documents. Pasting the *whole* handbook into every prompt works for a 20-page handbook, but not for thousands of pages
  of circulars, procedures and board papers. Finding the *right* few paragraphs for each question is retrieval, and that's
  Day 9.

**Your turn (2).** Write the three-sentence summary you'd give the head of member services: what fine-tuning achieved,
what it didn't, and what you recommend next. Put numbers from the verdict table in it.

*Your summary here.*

Save the full comparison and the verdict as CSV files: they're the evidence behind your summary, and Day 10's
governance pack will ask for them.

In [ ]:
Path("outputs").mkdir(exist_ok=True)
results.to_csv("outputs/heldout_comparison.csv", index=False)
verdict.to_csv("outputs/verdict.csv")
print("saved outputs/heldout_comparison.csv and outputs/verdict.csv")

## Looking back, looking ahead

**What you did in this lab.** You evaluated the fine-tuned model against the base model and against the base model with
the handbook pasted in, on 22 unseen staff questions: automatic key-fact scoring, results by question type, reading the
answers side by side, an LLM judge used with care, regression checks on general skills, and a verdict table with a
written summary.

**What you did today.** Day 8 was a complete fine-tuning project in its real proportions: most of the effort on data
(Lab 1), a short, efficient training run with LoRA (Lab 2), and an honest evaluation (Lab 3).

**The ideas to carry forward.**

- Fine-tuning changes *how* a model answers more reliably than *what facts* it knows.
- Giving the model the right text at question time decides whether the facts are right, and keeps them right when the
  rules change.
- Personal-data questions need a guardrail in the application, not just a better model.
- Evaluate on unseen questions, by type, against fair alternatives, and read the answers.
- Check for regressions every time.

**Tomorrow (Day 9).** Retrieval-augmented generation: for each question, find the few passages of the handbook and
circulars that answer it, have the model answer from those only with citations, and put it behind an internal API.